In [16]:
import pandas as pd

patient_df = pd.read_csv("data/raw/patient.csv")
doctor_df = pd.read_csv("data/raw/doctor.csv")
hospital_df = pd.read_csv("data/raw/hospital.csv")

raw_df = patient_df.merge(doctor_df, on="Patient_ID", how="left").merge(
    hospital_df, on="Hospital_ID", how="left"
)

raw_df.to_csv("data/raw/hospital_raw_data.csv", index=False)

In [17]:
df = pd.read_csv("data/raw/hospital_raw_data.csv")

missing_summary = pd.DataFrame(
    {
        "Missing_Count": df.isnull().sum(),
        "Missing_Percentage (%)": (df.isnull().sum() / len(df)) * 100,
    }
)

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(by="Missing_Count", ascending=False)

print("--- Missing Data Summary ---")
print(missing_summary)

structural_cols = ["Previous_Admission_Date"]
non_structural_missing = missing_summary.drop(
    index=structural_cols, errors="ignore"
)

overall_missing_pct = (
    non_structural_missing["Missing_Count"].sum()
    / (len(df) * (df.shape[1] - len(structural_cols)))
) * 100

print(
    f"\nOverall Dataset Missingness (Excluding Structural NaNs): {overall_missing_pct:.2f}%"
)

--- Missing Data Summary ---
                            Missing_Count  Missing_Percentage (%)
Previous_Admission_Date             44185               87.995141
Equipment_Used                       5052               10.061140
Doctor_Name                           782                1.557366
Insurance_Type                        770                1.533467
Age                                   762                1.517535
Patient_Satisfaction_Score            705                1.404019

Overall Dataset Missingness (Excluding Structural NaNs): 0.55%


In [8]:
import pandas as pd

df = pd.read_csv("data/processed/hospital_raw_data.csv")

df_clean = df.drop(columns=["Previous_Admission_Date"])

df_clean["Equipment_Used"] = df_clean["Equipment_Used"].fillna("None")

df_clean["Doctor_Name"] = df_clean["Doctor_Name"].fillna("Dr. Unassigned")

df_clean["Insurance_Type"] = df_clean["Insurance_Type"].fillna("Unspecified")

df_clean["Age"] = df_clean.groupby("Department")["Age"].transform(
    lambda x: x.fillna(x.median())
)

df_clean["Patient_Satisfaction_Score"] = df_clean.groupby("Department")[
    "Patient_Satisfaction_Score"
].transform(lambda x: x.fillna(round(x.mean(), 1)))

df_clean.to_csv("data/processed/hospital_cleaned.csv", index=False)

In [9]:
df = pd.read_csv("data/processed/hospital_cleaned.csv")

missing_summary = pd.DataFrame(
    {
        "Missing_Count": df.isnull().sum(),
        "Missing_Percentage (%)": (df.isnull().sum() / len(df)) * 100,
    }
)

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(by="Missing_Count", ascending=False)

print("--- Missing Data Summary ---")
print(missing_summary)

structural_cols = ["Previous_Admission_Date"]
non_structural_missing = missing_summary.drop(
    index=structural_cols, errors="ignore"
)

overall_missing_pct = (
    non_structural_missing["Missing_Count"].sum()
    / (len(df) * (df.shape[1] - len(structural_cols)))
) * 100

print(
    f"\nOverall Dataset Missingness (Excluding Structural NaNs): {overall_missing_pct:.2f}%"
)

--- Missing Data Summary ---
                Missing_Count  Missing_Percentage (%)
Equipment_Used           5052                10.06114

Overall Dataset Missingness (Excluding Structural NaNs): 0.36%


In [12]:
df = pd.read_csv("hospital_cleaned.csv")

exact_duplicates = df.duplicated().sum()
print(f"Total Exact Duplicate Rows: {exact_duplicates}")

patient_id_duplicates = df["Patient_ID"].duplicated().sum()
print(f"Duplicate Patient_IDs: {patient_id_duplicates}")

if exact_duplicates > 0:
    print("\n--- Exact Duplicate Rows ---")
    print(df[df.duplicated(keep=False)])

if patient_id_duplicates > 0:
    print("\n--- Rows with Duplicate Patient_IDs ---")
    print(df[df.duplicated(subset=["Patient_ID"], keep=False)])

Total Exact Duplicate Rows: 0
Duplicate Patient_IDs: 0


In [14]:
print(f"Final core table: {raw_df.shape}")
raw_df.head()

Final core table: (50213, 30)


,Patient_ID,Hospital_ID,Department,Admission_Date,Discharge_Date,Length_of_Stay_Days,Patient_Type,Admission_Type,Age,Gender,...,Equipment_Utilization_Rate_Pct,Treatment_Cost_USD,Insurance_Type,Discharge_Status,Patient_Satisfaction_Score,Department_Efficiency_Score,Doctor_ID,Doctor_Name,Hospital_Name,Region
0,PT000001,H001,Oncology,2024-12-27,2024-12-28,1,Inpatient,Newborn,6.0,Male,...,90.1,3214.04,Government Scheme,Discharged,4.0,100.0,DOC0100,Dr. Lee,City Care Hospital,North America
1,PT000002,H008,Pediatrics,2024-09-03,2024-09-08,5,Inpatient,Emergency,55.0,Female,...,78.1,3796.83,Self-Pay,Discharged,5.0,100.0,DOC0022,Dr. Patel,St. Mary's Hospital,North America
2,PT000003,H007,Gynecology,2024-08-08,2024-08-19,11,Inpatient,Urgent,60.0,Male,...,54.0,2889.63,Private Insurance,Discharged,3.0,93.7,DOC0069,Dr. Brown,Riverside Medical Center,South America
3,PT000004,H005,Gynecology,2024-08-14,2024-08-26,12,Inpatient,Elective,45.0,Male,...,86.2,6109.46,Private Insurance,Discharged,4.0,92.7,DOC0185,Dr. Brown,HealthPlus Hospital,Asia
4,PT000005,H005,Pediatrics,2024-09-28,2024-10-01,3,Emergency,Elective,16.0,Female,...,53.6,1972.45,Corporate Insurance,Discharged,1.0,100.0,DOC0204,Dr. Wang,HealthPlus Hospital,Asia


In [15]:
raw_df.columns.tolist()

['Patient_ID',
 'Hospital_ID',
 'Department',
 'Admission_Date',
 'Discharge_Date',
 'Length_of_Stay_Days',
 'Patient_Type',
 'Admission_Type',
 'Age',
 'Gender',
 'Diagnosis',
 'Readmission_Flag',
 'Previous_Admission_Date',
 'Bed_ID',
 'Bed_Type',
 'Total_Beds_In_Department',
 'Occupied_Beds_At_Admission',
 'Bed_Utilization_Rate_Pct',
 'Staff_On_Duty',
 'Equipment_Used',
 'Equipment_Utilization_Rate_Pct',
 'Treatment_Cost_USD',
 'Insurance_Type',
 'Discharge_Status',
 'Patient_Satisfaction_Score',
 'Department_Efficiency_Score',
 'Doctor_ID',
 'Doctor_Name',
 'Hospital_Name',
 'Region']